Esse fluxo é uma tentativa de utilizar o Kafka, a intenção é criar uma instancia que processe as mensagens recebidas por um script Python <br><br>
E depois disso o producer irá operar ao receber as mensagens <br><br>
comecei a fazer a configuração do AWS, mas obtive problemas por questão de bloqueio do AWS Academ. <br><br>
Optei por fazer a tentativa na maquina local com o Docker...<br><br>
 Posteriormente vou reproduzir em uma conta pessoal da AWS para conseguir fazer os testes

## Fluxo da AWS 

In [1]:
import os
import time
import boto3
import dotenv
from botocore.exceptions import ClientError

dotenv.load_dotenv()

True

Carregaremos as variaveis de ambiente e posteriormente faremos a conexão no boto3 com as variaveis carregadas

In [2]:
ID_CONTA = os.getenv("ID_CONTA")
AWS_REGION = os.getenv("AWS_REGION")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACESS_KEY")
#AWS_SESSION_TOKEN=os.getenv("aws_session_token") Token nao é necessario para fora do AWS Academy

In [3]:
# Criando a cessão 
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    #aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

# Inicializamos a conexão no Ec2 
ec2 = session.client('ec2')
print(f"Conectado com sucesso na região: {session.region_name}")


Conectado com sucesso na região: us-east-1


Criando uma VPC 

In [4]:
print("Criando VPC...")
vpc = ec2.create_vpc(
    CidrBlock='10.0.0.0/16',
    TagSpecifications=[
        {
            'ResourceType': 'vpc',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-vpc'}]
        }
    ]
)
vpc_id = vpc['Vpc']['VpcId']
print(f"VPC criada: {vpc_id}")
# Modifica os atributos da VPC para habilitar nomes DNS (necessário para o MSK)
ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={'Value': True})
ec2.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={'Value': True})

Criando VPC...


ClientError: An error occurred (UnauthorizedOperation) when calling the CreateVpc operation: You are not authorized to perform this operation. User: arn:aws:iam::703461918026:user/data-pipeline-user is not authorized to perform: ec2:CreateVpc on resource: arn:aws:ec2:us-east-1:703461918026:vpc/* because no identity-based policy allows the ec2:CreateVpc action. Encoded authorization failure message: my861PB1YQ0224xZGDfFZjHO-oMIDlQfQfgSXRHXlyeWienr5HbewFS1yZsepkZBaA59Lw4Rhd8HX3_KyMsa61yXmwjGtxz3FlOAZBScdunrvBZIHTV4nWdHGLTNR0dRO3Z8zkry5lWlYUccXc3_SMY0pc_rRJbkN-YaAXiTVhHhL7JwbnPywf-hzmkQtAhrp5goIA2YR79MxIGcJ1Rn_X1OlkC2tAPE_49ICsyGip8w53hmAZ2u6T0gf_iFoNfxV40_SJLTHdluJ1PNPM7K6XKVA6pVTUUo45_xX6wez3MEoKVG1J8d3jR7LAC4CNrNvQDAPgCyBlknNbLvAQSyKleJd9yxJtnm7kpULW7qMEAYsfZ88KuHJPe2fi8c2IM0YsyHd0jsdHBzmPBiwi1Swu4VrstOMPvmLX02VIEhg6QLrUYnqW1v3pN5Q3Fc6uX8DP7sXWUdA2TooXpTnxM46Iftp1EdpM7lQ8CMLe3Nu0un2XxYVHQItSlZT7zv6UC0gGlxoJwLFwS9gT6vFFwTUF9A62Ju

verificando zonas de disponibilidade e subnets 

Zonas de Disponibilidade (AZs): São os Data Centers físicos existentes dentro da região. Cada um desses grupos de data centers é uma Zona de Disponibilidade. 
Para rodar o Kafka optamos por usar duas zonas de disponibilidade diferentes, ou seja dois Brokers. Se uma deixar de funcionar a outra assume. 

já os subnets sao as divisões da VPC (Virtual Private Cloud), Se as zonas de disponibilidade são os datacenters dentro de uma região, os subnets são as salas dentro das zonas de disponibilidade


> Essa parte do código é utilizada para verificar as zonas de disponibilidade, como fiz no academy tive problema para recupera-las
por conta de bloqueio de autorização 
encontrei o nome das zonas disponiveis na região que o academy forneceu e utilizei aqui hardcode mesmo 

In [ ]:
#--------
# azs = ec2.describe_availability_zones(
#     Filters=[{'Name': 'state', 'Value': 'available'}]
# )['AvailabilityZones']
az1 = f"{AWS_REGION}a"  # Resultará em 'us-east-1a'
az2 = f"{AWS_REGION}b"  # Resultará em 'us-east-1b'
print(f"Zonas de disponibilidade definidas manualmente: {az1} e {az2}")

Criando os subnet

In [ ]:
print(f"Criando Subnet 1 em {az1}...")
subnet1 = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='10.0.1.0/24',
    AvailabilityZone=az1,
    TagSpecifications=[
        {
            'ResourceType': 'subnet',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-subnet-1'}]
        }
    ]
)
subnet1_id = subnet1['Subnet']['SubnetId']

print(f"Criando Subnet 2 em {az2}...")
subnet2 = ec2.create_subnet(
    VpcId=vpc_id,
    CidrBlock='10.0.2.0/24',
    AvailabilityZone=az2,
    TagSpecifications=[
        {
            'ResourceType': 'subnet',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-subnet-2'}]
        }
    ]
)
subnet2_id = subnet2['Subnet']['SubnetId']
print(f"Subnets criadas: {subnet1_id} e {subnet2_id}")


Por fim configurar o Internet Gateway, são os nossos acessos as zonas as nossas subnets 

**Route Table** é a definição das nossas rotas de rede para as subnet

In [ ]:
# Criando e associando o Internet Gateway
print("Criando Internet Gateway...")
igw = ec2.create_internet_gateway(
    TagSpecifications=[
        {
            'ResourceType': 'internet-gateway',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-igw'}]
        }
    ]
)
igw_id = igw['InternetGateway']['InternetGatewayId']
ec2.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
print(f"Internet Gateway anexado: {igw_id}")
# Criando uma Route Table
print("Criando Tabela de Roteamento...")
rt = ec2.create_route_table(
    VpcId=vpc_id,
    TagSpecifications=[
        {
            'ResourceType': 'route-table',
            'Tags': [{'Key': 'Name', 'Value': 'fiap-msk-rt'}]
        }
    ]
)
rt_id = rt['RouteTable']['RouteTableId']
# Adicionando a rota de saída padrão (0.0.0.0/0) e apontando para o Internet Gateway (igw_id)
ec2.create_route(
    RouteTableId=rt_id,
    DestinationCidrBlock='0.0.0.0/0',
    GatewayId=igw_id
)
# Por fim associar a Route Table com ambas as subnets
ec2.associate_route_table(SubnetId=subnet1_id, RouteTableId=rt_id)
ec2.associate_route_table(SubnetId=subnet2_id, RouteTableId=rt_id)
print(f"Tabela de Roteamento {rt_id} associada às subnets.")
print("\n✅ Infraestrutura de rede inicial criada com sucesso!")

Conectar ao S3 

In [ ]:
s3 = session.client('s3')
bucket_name = os.getenv("BUCKET_NAME")
print(f"Nome do bucket a ser criado: {bucket_name}")

In [ ]:
s3 = boto3.client('s3')
nome_do_bucket = os.getenv("BUCKET_NAME")

response = s3.list_buckets()
buckets_existentes = [bucket['Name'] for bucket in response['Buckets']]

try:
    # verifica a existencia do bucket da Fiap na AWS conectada
    s3.head_bucket(Bucket=nome_do_bucket)
    print(f"Bucket '{nome_do_bucket}' já existe.")
except ClientError as e:
    # Se der erro 404 (Not Found), significa que o bucket não existe então iremos cria-lo
    error_code = e.response['Error']['Code']
    if error_code == '404':
        s3.create_bucket(Bucket=nome_do_bucket)
        print(f"Bucket '{nome_do_bucket}' criado com sucesso.")
    else:
        # Outro erro
        print(f"Erro ao acessar o bucket: {e}")

# Ativar versionamento no bucket (boa prática e exigido pelo laboratório do Glue)
print("Ativando versionamento no bucket...")
s3.put_bucket_versioning(
    Bucket=bucket_name,
    VersioningConfiguration={'Status': 'Enabled'}
)
print("[INFO] Versionamento ativado com sucesso!")

Criando o security Group, para servir como um Firewall de controle de entrada e saida. 
iremos liberar as portas
9092 para o Kafka (PLAINTEXT)  <br><br>
O professor utilizou a porta 2181 Para o Zookeper (gerenciador dos Cluster), mas a partir da versão 3.X do Kafka o Zookeper deixa de ser necessário <br> <br>
optei por tentar esse caminho 

In [ ]:
# Nome do Security Group configurado no .env
sg_name = os.getenv("SG_NAME")

print(f"Nome do Security Group a ser criado: {sg_name}")

try:
    # Criando SG na VPC 
    print("Criando o Security Group...")
    sg = ec2.create_security_group(
        GroupName=sg_name,
        Description="SG for MSK Provisioned lab",
        VpcId=vpc_id # Variável contendo o ID da VPC criada anteriormente
    )
    sg_id = sg['GroupId']
    print(f"[SUCESSO] SG criado com ID: {sg_id}")

    # Configurando as portas para acesso
    print("[INFO] Adicionando regras de entrada para o Kafka...")
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpPermissions=[
            # Porta 9092 para a rede interna da VPC (10.0.0.0/16)
            {
                'IpProtocol': 'tcp',
                'FromPort': 9092,
                'ToPort': 9092,
                'IpRanges': [{'CidrIp': '10.0.0.0/16', 'Description': 'porta para o Kafka PLAINTEXT'}]
            },
            # # Porta 2181 Utilizada para o Zookeper, foi removida 
            # {
            #     'IpProtocol': 'tcp',
            #     'FromPort': 2181,
            #     'ToPort': 2181,
            #     'IpRanges': [{'CidrIp': '10.0.0.0/16', 'Description': 'ZooKeeper'}]
            # },
            # Permissão interna (auto-referência) necessária para os workers do Glue
            {
                'IpProtocol': 'tcp',
                'FromPort': 0,
                'ToPort': 65535,
                'UserIdGroupPairs': [{'GroupId': sg_id, 'Description': 'Porta para trafego interno do Glue/Kafka'}]
            }
        ]
    )
    print("[SUCESSO] Regras de firewall configuradas com sucesso!")

except ec2.exceptions.ClientError as e:
    # Tratamento caso o Security Group já tenha sido criado anteriormente
    if 'InvalidGroup.Duplicate' in str(e):
        print(f"[INFO] O Security Group '{sg_name}' já existe. Recuperando ID...")
        existing_sgs = ec2.describe_security_groups(
            Filters=[
                {'Name': 'group-name', 'Values': [sg_name]},
                {'Name': 'vpc-id', 'Values': [vpc_id]}
            ]
        )
        sg_id = existing_sgs['SecurityGroups'][0]['GroupId']
        print(f"[SUCESSO] ID do Security Group existente recuperado: {sg_id}")
    else:
        print(f"[ERRO] : {e}")

# Mantém o ID salvo na memória para o próximo passo
print(f"\nSG_ID ativo: {sg_id}")


In [ ]:
# #Deletar o SG 
# ec2.delete_security_group(GroupId=sg_id)
# print("SG Deletado.")

Inicializando o Kafka 

In [ ]:
# Inicializar o cliente do Kafka (MSK)
kafka = session.client('kafka')

cluster_name = os.getenv("CLUSTER_NAME")
print(f"Iniciando a criação do cluster MSK (KRaft): {cluster_name}")

try:
    response = kafka.create_cluster(
        ClusterName=cluster_name,
        KafkaVersion="3.7.x.kraft",  # kraft ativa o modo KRaft (Sem ZooKeeper)
        NumberOfBrokerNodes=2,       # broker em cada subnet
        BrokerNodeGroupInfo={
            'InstanceType': 'kafka.t3.small',
            'ClientSubnets': [subnet1_id, subnet2_id], # Criadas no Passo 0
            'SecurityGroups': [sg_id],                 # Criado no Passo 2
            'StorageInfo': {
                'EbsStorageInfo': {
                    'VolumeSize': 10  # 10 GB de armazenamento
                }
            }
        },
        EncryptionInfo={
            'EncryptionInTransit': {
                'ClientBroker': 'PLAINTEXT', # Tráfego interno de dados sem criptografia
                'InCluster': False
            }
        }
    )
    cluster_arn = response['ClusterArn']
    print(f"[SUCESSO] Cluster MSK enviado para criação com sucesso!")
    print(f"Cluster ARN: {cluster_arn}")
    print("\n[INFO] O cluster leva cerca de 15 a 20 minutos para ficar 'ACTIVE'.")

except kafka.exceptions.BadRequestException as e:
    if 'already exists' in str(e):
        print(f"[INFO] O cluster '{cluster_name}' já existe. Recuperando detalhes...")
        clusters = kafka.list_clusters(ClusterNameFilter=cluster_name)['ClusterInfoList']
        cluster_arn = clusters[0]['ClusterArn']
        print(f"[SUCESSO] ARN do cluster recuperado: {cluster_arn}")
    else:
        print(f"[ERRO] falha ao criar o cluster: {e}")
except Exception as e:
    print(f"[ERRO]: {e}")


Esse fluxo foi feito pelo professor na aula gravada [LINK](https://zoom.us/rec/play/iLSLbM9bB2XKhY9FEyhrviWsoJ1AlJAIcEQ7Xk9dJzpkQz6CZprnlFd5lkmHJ1M9FOva-P_yd_zmRAga.2hhqpObycOC0ZS92?accessLevel=meeting&canPlayFromShare=true&from=share_recording_detail&startTime=1781647585000&oldStyle=true&componentName=rec-play&originRequestUrl=https%3A%2F%2Fzoom.us%2Frec%2Fshare%2FB8RCJOVlnPMmobZvMgXKdPyePQeA7Kj6R_fXDOxGS-TebFC36ngCgCELqnAG0-Jv.dOgH2tzEroMkVNZ4%3FstartTime%3D1781647585000)

Utilizei como base para gerar todo o código e adaptar para script em python em vez do script padrão em CLI do AWS, assim os demais colegas pudessem utilizar o fluxo em suas maquinas sem grande dificuldade

**ATENÇÃO AO LIGAR O CLUSTER ELE PROVISIONARÁ UM CUSTO DE U$ 0,07 POR HORA PARA CADA BROKER, <br> ENTÃO É IMPORTANTE DESLIGAR O CLUSTER AO FINAL DOS TESTES**

